# Analysis 3 — Staggered DiD across LNG supplier countries

**Causal question**: Did locking in via a long-term Japanese SPA cause a higher and more persistent share of Japan's LNG imports than later spot-market entry would have produced?

**Design**: Country-year panel of major LNG suppliers to Japan, 1970–2010. Treatment cohorts staggered by year of first LNG cargo under a long-term Japanese SPA: Brunei 1972, Indonesia 1977, UAE 1977, Malaysia 1983, Australia 1989, Qatar 1997, Oman 2000. USA and Russia included as never-treated/late-treated controls.

**Outcome**: `value_share_of_japan_total` — country c's share of Japan's total LNG imports in year t (HS 271111 from 1996+, HS 2711 1988–1995, SITC 341 1970–1987).

**Estimator**: Callaway-Sant'Anna (2021) via the `differences` package. Sun-Abraham (2021) interaction-weighted TWFE is the fallback. Event-time dummies are named `dlead{n}` / `dlag{n}` to avoid patsy parsing problems with hyphens.

**Sample window**: 1970–2010 (pre-Fukushima).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

REPO = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
PROC = REPO / 'data' / 'processed'
MAN = REPO / 'data' / 'manual'

panel = pd.read_csv(PROC / 'japan_lng_supplier_panel.csv')
cohorts = pd.read_csv(MAN / 'lng_supplier_spa_cohorts.csv')

# Pre-1988 SITC 341 includes non-LNG manufactured gas for some countries.
# Australia exported NO LNG to Japan before 1989 (NWS first cargo Jul 1989);
# zero out any pre-1989 noise in the AUS series so the event-time-0 estimate
# isn't contaminated.
panel.loc[(panel.partner_iso == 'AUS') & (panel.year < 1989),
          ['value_usd', 'net_weight_kg', 'value_share_of_japan_total', 'weight_share_of_japan_total']] = 0

print('panel shape:', panel.shape)
print(panel.head())
print('\nCohort manifest:')
print(cohorts[['country', 'iso3', 'first_spa_signed_year', 'first_cargo_year', 'treatment_cohort']])

## 1. Raw share series — visual sanity check

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for iso, grp in panel.groupby('partner_iso'):
    if grp['value_share_of_japan_total'].notna().sum() < 3:
        continue
    ax.plot(grp['year'], grp['value_share_of_japan_total'], label=iso, marker='o', ms=3)
    cohort_year = grp['first_cargo_year'].dropna().iloc[0] if grp['first_cargo_year'].notna().any() else None
    if cohort_year is not None:
        ax.axvline(cohort_year, color='grey', alpha=0.15, lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Share of Japan LNG imports (value)')
ax.legend(loc='upper left', ncol=2, fontsize=8)
ax.set_title('Country share of Japan LNG imports, 1970-2010')
plt.tight_layout(); plt.show()

## 2. Callaway-Sant'Anna (2021) ATT(g, t)

Treatment year = `first_cargo_year`. Outcome = `value_share_of_japan_total`. Never-treated control group = countries with no treatment year recorded.

In [ ]:
try:
    from differences import ATTgt
    HAVE_DIFFERENCES = True
except ImportError:
    HAVE_DIFFERENCES = False
    print('differences not installed. Install with `pip install differences`.')

if HAVE_DIFFERENCES:
    df = panel.copy()
    df['cohort'] = df['first_cargo_year'].fillna(0).astype(int)
    df = df.dropna(subset=['value_share_of_japan_total'])
    df = df.set_index(['partner_iso', 'year'])
    try:
        attgt = ATTgt(data=df, cohort_name='cohort', freq='Y', anticipation_periods=0)
        attgt.fit(formula='value_share_of_japan_total', control_group='never_treated')
        print(attgt.results_)
    except Exception as e:
        print(f'differences fit failed: {type(e).__name__}: {e}')
        print('Falling back to Sun-Abraham TWFE below.')

## 3. Sun-Abraham (2021) interaction-weighted TWFE

Event-time dummies `dlead{n}` (year before treatment) and `dlag{n}` (year of treatment + n). Reference period: `dlead1` (one year before treatment), omitted.

In [ ]:
EVENT_LEADS = range(1, 11)   # 10 leads
EVENT_LAGS = range(0, 16)    # 16 lags (0 through 15)

df = panel.dropna(subset=['value_share_of_japan_total']).copy()
df['cohort'] = df['first_cargo_year'].fillna(0).astype(int)
df['event_time'] = df['year'] - df['first_cargo_year']
treated = df[df['cohort'] > 0].copy()
treated = treated[treated['event_time'].between(-max(EVENT_LEADS), max(EVENT_LAGS))]

for n in EVENT_LEADS:
    if n == 1: continue  # omitted reference
    treated[f'dlead{n}'] = (treated['event_time'] == -n).astype(int)
for n in EVENT_LAGS:
    treated[f'dlag{n}'] = (treated['event_time'] == n).astype(int)

regressors = ([f'dlead{n}' for n in EVENT_LEADS if n != 1]
              + [f'dlag{n}' for n in EVENT_LAGS])
rhs = ' + '.join(regressors)
formula = f'value_share_of_japan_total ~ {rhs} + C(partner_iso) + C(year)'
model = smf.ols(formula, data=treated).fit(cov_type='cluster',
                                            cov_kwds={'groups': treated['partner_iso']})
print(model.summary().tables[1])

In [ ]:
coefs, ses, evs = [], [], []
for n in EVENT_LEADS:
    if n == 1:
        coefs.append(0.0); ses.append(0.0); evs.append(-1); continue
    name = f'dlead{n}'
    if name in model.params.index:
        coefs.append(model.params[name]); ses.append(model.bse[name]); evs.append(-n)
for n in EVENT_LAGS:
    name = f'dlag{n}'
    if name in model.params.index:
        coefs.append(model.params[name]); ses.append(model.bse[name]); evs.append(n)
order = np.argsort(evs)
evs = np.array(evs)[order]; coefs = np.array(coefs)[order]; ses = np.array(ses)[order]

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(evs, coefs, yerr=1.96*ses, fmt='o-', capsize=3)
ax.axhline(0, color='black', lw=0.5)
ax.axvline(-0.5, color='red', lw=0.5, ls='--', label='Treatment year')
ax.set_xlabel('Event time (years from first cargo)')
ax.set_ylabel('Effect on country share of Japan LNG imports')
ax.set_title('Event-time ATT — staggered DiD across LNG suppliers')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Australia vs Qatar — SPA-locked-in vs late-mover comparison

If long-term SPAs lock in flows, Australia (1989, with 5.0 Mtpa locked in 20-yr SPAs to 8-utility consortium) should plateau higher than Qatar (1997, comparable 4.0 Mtpa).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for iso, color in [('AUS', 'tab:blue'), ('QAT', 'tab:orange')]:
    sub = panel[panel.partner_iso == iso].copy()
    sub = sub.dropna(subset=['value_share_of_japan_total', 'event_time'])
    sub = sub[sub['event_time'].between(-5, 15)]
    ax.plot(sub['event_time'], sub['value_share_of_japan_total'], '-o',
            color=color, label=iso, ms=4)
ax.axvline(0, color='red', lw=0.5, ls='--')
ax.set_xlabel('Event time (years from first cargo)')
ax.set_ylabel('Share of Japan LNG imports')
ax.set_title('Australia (1989) vs Qatar (1997) post-entry trajectories')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Robustness — SPA-signing year as alternative treatment definition

SPA signing precedes first cargo by 3–5 years. Re-run with `first_spa_signed_year`.

In [ ]:
alt = panel.merge(cohorts.rename(columns={'iso3': 'partner_iso'})[['partner_iso', 'first_spa_signed_year']],
                  on='partner_iso', how='left', suffixes=('', '_y'))
alt['event_time_spa'] = alt['year'] - alt['first_spa_signed_year']
alt['cohort_spa'] = alt['first_spa_signed_year'].fillna(0).astype(int)
alt = alt.dropna(subset=['value_share_of_japan_total', 'event_time_spa'])
alt = alt[alt['cohort_spa'] > 0]
alt = alt[alt['event_time_spa'].between(-max(EVENT_LEADS), max(EVENT_LAGS))]
for n in EVENT_LEADS:
    if n == 1: continue
    alt[f'dlead{n}'] = (alt['event_time_spa'] == -n).astype(int)
for n in EVENT_LAGS:
    alt[f'dlag{n}'] = (alt['event_time_spa'] == n).astype(int)
m2 = smf.ols(f'value_share_of_japan_total ~ {rhs} + C(partner_iso) + C(year)', data=alt).fit(
    cov_type='cluster', cov_kwds={'groups': alt['partner_iso']})
print('SPA-signing-year treatment definition:')
print(m2.summary().tables[1])

## 6. Synthetic control footnote — Australia 1989 entry

With only Brunei (17 pretreatment years), Indonesia (12 years), and Malaysia (6 years) as feasible donors, synthetic control is fragile. Report as a sensitivity check, not a primary result.

Donors: {Brunei, Indonesia, Malaysia}, convex weights summing to 1.

In [ ]:
from scipy.optimize import minimize

DONORS = ['BRN', 'IDN', 'MYS']
wide = panel.pivot_table(index='year', columns='partner_iso',
                         values='value_share_of_japan_total', aggfunc='first')
pre_years = list(range(1977, 1989))
Y_pre = wide.loc[pre_years, 'AUS'].fillna(0).values
X_pre = wide.loc[pre_years, DONORS].fillna(0).values

def loss(w):
    return float(np.sum((Y_pre - X_pre @ w) ** 2))
cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
bnds = [(0, 1)] * len(DONORS)
res = minimize(loss, x0=np.ones(len(DONORS))/len(DONORS), bounds=bnds, constraints=cons)
weights = dict(zip(DONORS, res.x))
print('Synthetic-Australia weights:', weights)

post_years = list(range(1989, 2011))
Y_post = wide.loc[post_years, 'AUS'].fillna(0).values
Y_synth = wide.loc[post_years, DONORS].fillna(0).values @ res.x
gap = Y_post - Y_synth
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(post_years, Y_post, 'o-', label='Actual Australia')
ax.plot(post_years, Y_synth, 's-', label='Synthetic Australia')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Share of Japan LNG imports')
ax.set_title('Synthetic control: Australia vs counterfactual (fragile — 3 donors only)')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Average post-1989 treatment effect (gap): {gap.mean():.4f}')

## 7. Verification checks

1. ATT(Australia, e=15) — 2004 — should match JOGMEC-reported ~17% Australian share of Japan LNG.
2. ATT(Australia, e=0) should be small.
3. Aggregated ATT across cohorts should be positive.

In [ ]:
aus_2004 = panel[(panel.partner_iso=='AUS') & (panel.year==2004)]['value_share_of_japan_total']
print('Actual Australia share of Japan LNG in 2004:', aus_2004.iloc[0] if len(aus_2004) else 'N/A')
print('Expected (JOGMEC ~2004): ~0.17')